In [ ]:
import sys
sys.path.append('/host/c/Users/ROG/Documents/GitHub')
import nibabel as nb
import glob
import os
import lpips
import torch
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

import Diffusion_denoising_thin_slice.functions_collection as ff
import Diffusion_denoising_thin_slice.Build_lists.Build_list as Build_list
import Diffusion_denoising_thin_slice.Data_processing as Data_processing

In [ ]:
build_sheet = Build_list.Build_thinsliceCT(os.path.join('/host/d/Data/brain_CT/Patient_lists/fixedCT_static_simulation_train_test_gaussian_xjtlu.xlsx'))
_,patient_id_list,patient_subid_list,random_num_list, condition_list, x0_list = build_sheet.__build__(batch_list = [5])
n = ff.get_X_numbers_in_interval(total_number = patient_id_list.shape[0],start_number = 0,end_number = 1, interval = 2)

In [ ]:
def calc_mae_with_ref_window(img, ref, vmin, vmax):
    maes = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        mae = np.sum(np.abs(slice_img - slice_ref) * mask) / np.sum(mask)
        maes.append(mae)
    return np.mean(maes), np.std(maes)

In [ ]:
def calc_ssim_with_ref_window(img, ref, vmin, vmax):
    ssims = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        _, ssim_map = structural_similarity(slice_img, slice_ref, data_range=vmax - vmin, full=True)
        ssim = np.sum(ssim_map * mask) / np.sum(mask)
        ssims.append(ssim)
    return np.mean(ssims), np.std(ssims)

In [ ]:
def calc_lpips(imgs1, imgs2, vmin, vmax):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loss_fn = lpips.LPIPS().to(device)
    lpipss = []
    for slice_num in range(0, imgs1.shape[-1]):
        slice1 = imgs1[:,:,slice_num]
        slice2 = imgs2[:,:,slice_num]
        slice1 = np.clip(slice1, vmin, vmax).astype(np.float32)
        slice2 = np.clip(slice2, vmin, vmax).astype(np.float32)
        slice1 = (slice1 - vmin) / (vmax - vmin) * 2 - 1
        slice2 = (slice2 - vmin) / (vmax - vmin) * 2 - 1
        slice1 = np.stack([slice1, slice1, slice1], axis=-1)
        slice2 = np.stack([slice2, slice2, slice2], axis=-1)
        slice1 = np.transpose(slice1, (2, 0, 1))[np.newaxis, ...]
        slice2 = np.transpose(slice2, (2, 0, 1))[np.newaxis, ...]
        slice1 = torch.from_numpy(slice1).to(device)
        slice2 = torch.from_numpy(slice2).to(device)
        lpips_val = loss_fn(slice1, slice2)
        lpipss.append(lpips_val.item())
    return np.mean(lpipss), np.std(lpipss)

In [ ]:
## IMF metric calculations
import shutil

imf_trial_name = 'imf_unsupervised_gaussian_brainCT'
imf_epoch = 30

results = []
save_collection = True
for i in range(0, n.shape[0]):
    patient_id = patient_id_list[n[i]]
    patient_subid = patient_subid_list[n[i]]
    random_n = random_num_list[n[i]]

    print(patient_id, patient_subid, random_n)

    save_folder = os.path.join('/host/d/projects/denoising/pred_collections/brain_CT_imf', patient_id, patient_subid)
    ff.make_folder([os.path.dirname(save_folder), save_folder])

    # reference image
    gt_file = os.path.join('/host/d/projects/denoising/models', imf_trial_name, 'pred_images', patient_id, patient_subid, 'random_'+str(random_n), 'epoch'+str(imf_epoch)+'_1/gt_img.nii.gz')
    gt_img = nb.load(gt_file).get_fdata()
    if save_collection:
        shutil.copy(gt_file, os.path.join(save_folder, 'gt_img.nii.gz'))

    # noisy image
    condition_file = os.path.join('/host/d/projects/denoising/models', imf_trial_name, 'pred_images', patient_id, patient_subid, 'random_'+str(random_n), 'epoch'+str(imf_epoch)+'_1/condition_img.nii.gz')
    condition_img = nb.load(condition_file).get_fdata()
    if save_collection:
        shutil.copy(condition_file, os.path.join(save_folder, 'condition_img.nii.gz'))

    # IMF single inference
    imf_file = os.path.join('/host/d/projects/denoising/models', imf_trial_name, 'pred_images', patient_id, patient_subid, 'random_'+str(random_n), 'epoch'+str(imf_epoch)+'_1/pred_img.nii.gz')
    imf_img = nb.load(imf_file).get_fdata()
    if save_collection:
        shutil.copy(imf_file, os.path.join(save_folder, 'imf_img.nii.gz'))

    # IMF avg10
    imf_avg10_file = os.path.join('/host/d/projects/denoising/models', imf_trial_name, 'pred_images', patient_id, patient_subid, 'random_'+str(random_n), 'epoch'+str(imf_epoch)+'avg/pred_img_scans10.nii.gz')
    imf_avg10_img = nb.load(imf_avg10_file).get_fdata()
    if save_collection:
        shutil.copy(imf_avg10_file, os.path.join(save_folder, 'imf_avg10_img.nii.gz'))

    # IMF avg20
    imf_avg20_file = os.path.join('/host/d/projects/denoising/models', imf_trial_name, 'pred_images', patient_id, patient_subid, 'random_'+str(random_n), 'epoch'+str(imf_epoch)+'avg/pred_img_scans20.nii.gz')
    imf_avg20_img = nb.load(imf_avg20_file).get_fdata()
    if save_collection:
        shutil.copy(imf_avg20_file, os.path.join(save_folder, 'imf_avg20_img.nii.gz'))

    ## calculate metrics
    vmin = 0
    vmax = 100

    # MAE
    mae_condition, mae_condition_std = calc_mae_with_ref_window(condition_img, gt_img, vmin, vmax)
    mae_imf, mae_imf_std = calc_mae_with_ref_window(imf_img, gt_img, vmin, vmax)
    mae_imf_avg10, mae_imf_avg10_std = calc_mae_with_ref_window(imf_avg10_img, gt_img, vmin, vmax)
    mae_imf_avg20, mae_imf_avg20_std = calc_mae_with_ref_window(imf_avg20_img, gt_img, vmin, vmax)

    # SSIM
    ssim_condition, ssim_condition_std = calc_ssim_with_ref_window(condition_img, gt_img, vmin, vmax)
    ssim_imf, ssim_imf_std = calc_ssim_with_ref_window(imf_img, gt_img, vmin, vmax)
    ssim_imf_avg10, ssim_imf_avg10_std = calc_ssim_with_ref_window(imf_avg10_img, gt_img, vmin, vmax)
    ssim_imf_avg20, ssim_imf_avg20_std = calc_ssim_with_ref_window(imf_avg20_img, gt_img, vmin, vmax)

    # LPIPS
    lpips_condition, _ = calc_lpips(condition_img, gt_img, vmin, vmax)
    lpips_imf, _ = calc_lpips(imf_img, gt_img, vmin, vmax)
    lpips_imf_avg10, _ = calc_lpips(imf_avg10_img, gt_img, vmin, vmax)
    lpips_imf_avg20, _ = calc_lpips(imf_avg20_img, gt_img, vmin, vmax)

    print('condition: ', mae_condition, ssim_condition, lpips_condition)
    print('imf: ', mae_imf, ssim_imf, lpips_imf)
    print('imf_avg10: ', mae_imf_avg10, ssim_imf_avg10, lpips_imf_avg10)
    print('imf_avg20: ', mae_imf_avg20, ssim_imf_avg20, lpips_imf_avg20)

    results.append([patient_id, patient_subid, random_n,
                    mae_condition, mae_imf, mae_imf_avg10, mae_imf_avg20,
                    ssim_condition, ssim_imf, ssim_imf_avg10, ssim_imf_avg20,
                    lpips_condition, lpips_imf, lpips_imf_avg10, lpips_imf_avg20
                    ])

    dd = pd.DataFrame(results, columns=['patient_id','patient_subid','random_n',
        'mae_condition','mae_imf','mae_imf_avg10','mae_imf_avg20',
        'ssim_condition','ssim_imf','ssim_imf_avg10','ssim_imf_avg20',
        'lpips_condition','lpips_imf','lpips_imf_avg10','lpips_imf_avg20'
        ])
    dd.to_excel('/host/d/projects/denoising/results/brainCT_imf_results.xlsx', index=False)

In [ ]:
# print summary
print('\n===== Summary =====')
print(dd[['mae_condition','mae_imf','mae_imf_avg10','mae_imf_avg20']].mean())
print()
print(dd[['ssim_condition','ssim_imf','ssim_imf_avg10','ssim_imf_avg20']].mean())
print()
print(dd[['lpips_condition','lpips_imf','lpips_imf_avg10','lpips_imf_avg20']].mean())